# GPU-Accelerated Thermal Diffusion - benchmark notebook (Person 3)
Runs top to bottom on a **Colab GPU runtime**: *Runtime -> Change runtime type -> T4 GPU*.
Order: setup -> environment -> correctness -> smoke test -> full benchmark -> analysis -> download.
Run everything in **one session** so all numbers share the same hardware. The benchmark step is resumable: if the session dies, reconnect, rerun the setup cell, then rerun the benchmark cell.

## 1. Setup

In [ ]:
!nvidia-smi
!nvcc --version
import os
REPO = 'https://github.com/Keerthi291890/GPU-Accelerated-Thermal-Diffusion-.git'
if not os.path.exists('proj'):
    !git clone {REPO} proj
%cd proj
!git pull
!git log --oneline | head -5
!pip -q install pytest

## 2. Environment capture (saved to `results/env/`)

In [ ]:
!python benchmarks/run_benchmarks.py --env-only
!cat results/env/environment.json

## 3. Correctness - MUST pass before any timing is trusted
`pytest` covers the maths and the CUDA kernels on grids that are not multiples of the block size. `validate_all.py` runs the full matrix and writes `results/correctness_full.csv`.

In [ ]:
!python -m pytest tests -q

In [ ]:
!python benchmarks/validate_all.py

## 4. Smoke test (about 1-2 minutes) - confirms the harness works on this GPU

In [ ]:
!python benchmarks/run_benchmarks.py --out results/smoke --sizes 512,1024 --only scaling --rounds 1 --cuda-reps 3
!python benchmarks/analyze.py --out results/smoke

## 5. Full benchmark
Defaults: sizes 128-4096, K=1000, 5 block shapes x naive/tiled, step sweep K=10...10000, OpenMP thread sweep, 3 rounds x 7 CUDA repetitions.
Add `--sizes 128,256,512,1024,2048,4096,8192` for the optional 8192 grid (CPU runs get slow). Check the plan first with `--dry-run`.

In [ ]:
!python benchmarks/run_benchmarks.py --dry-run | head -3

In [ ]:
!python benchmarks/run_benchmarks.py

## 6. Analysis - tables and plots

In [ ]:
!python benchmarks/analyze.py
print(open('results/summary_tables.md').read())

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/plots/*.png')):
    print(p); display(Image(p))

## 7. Package results for download (commit them to git as well)

In [ ]:
!zip -rq results_colab.zip results/raw_timings.csv results/summary.csv results/summary_tables.md results/correctness_full.csv results/env results/plots
from google.colab import files
files.download('results_colab.zip')